# Spring 26 LLMs Workshop

March 2, 2026
Part 2, Phase 2
Retrieval Augmented Generation (RAG)

We build a local QA system using Gemini and Chroma DB.

## Phase 2 overview

This Phase 2 notebook is about RAG (Retrieval Augmented Generation).

RAG has two main parts:
(A) Knowledge base build pipeline: load documents, split them, create embeddings, and index/store these chunks in a vector database (Chroma).
(B) Retrieval Augmented Generation pipeline: for each user query, retrieve relevant chunks from that knowledge base and pass them as context to the LLM to generate grounded answers.

We use LangChain because it provides robust components for loaders, splitters, embeddings, vector stores, retrievers, prompt templates, and LLM calls.

## Install libraries

In [ ]:
# %pip -q install langchain langchain-community langchain-google-genai chromadb pypdf python-dotenv

## Imports and config

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate

DOCS_DIR = "./documents"
CHROMA_DIR = "./chroma_db"
COLLECTION_NAME = "spring26_papers"
TOP_K = 5
GEMINI_LLM_MODEL = "gemini-2.5-flash"
GEMINI_EMBED_MODEL = "models/gemini-embedding-001"

memory = {"turns": []}


In [ ]:
def dict_to_str(memory_dict, last_n=3):
    """Convert memory['turns'] into prompt text. Supports both schemas:
    1) {"role": "...", "content": "..."}
    2) {"user": "...", "assistant": "..."}
    """
    turns = (memory_dict or {}).get("turns", [])
    if not turns:
        return "(No previous history)"

    turns = turns[-last_n:]  # keep it short
    lines = []

    for t in turns:
        # Schema A: role/content
        if "role" in t and "content" in t:
            role = (t.get("role") or "unknown").upper()
            content = (t.get("content") or "").strip()
            if content:
                lines.append(f"{role}: {content}")
            continue

        # Schema B: user/assistant
        u = (t.get("user") or "").strip()
        a = (t.get("assistant") or "").strip()
        if u:
            lines.append(f"USER: {u}")
        if a:
            lines.append(f"ASSISTANT: {a}")

    return "\n".join(lines) if lines else "(No previous history)"

In [ ]:
memory

In [ ]:
def rewrite_query(query: str) -> str:
    """TODO: later add query rewriting using a lighter LLM"""
    
    rewrite_query_prompt = PromptTemplate(
        input_variables=["memory", "query"],
        template="""You are a query rewriting tool for RAG retrieval.
Your job is ONLY to rewrite the user's query to improve document retrieval.
DO NOT answer the question.

What to do:
- Use the conversation history to infer what the follow-up refers to.
- Extract and append the most important keywords from memory: paper title, key concept names, method names, equations, section terms, dataset names.
- Rewrite the query as a standalone search query so embeddings retrieve the correct chunks.
- Keep it short and keyword-rich (1 to 2 lines max).
- Output ONLY the rewritten query. No explanation.

--- CONVERSATION HISTORY ---
{memory}

--- USER QUERY ---
{query}

Rewritten retrieval query:

    --- CONVERSATION HISTORY ---
    {memory}

    --- USER QUESTION ---
    {query}

    Write a clear, structured answer.
    """
    )
    rewrite_llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.1,
        max_output_tokens=1000
    )
    memory_str=dict_to_str(memory)
    print(memory_str)
    chain = rewrite_query_prompt | rewrite_llm
    rewritten = chain.invoke({"memory": memory_str, "query": query}).content.strip()
    query=rewritten     
    
    return query

## API key

In [ ]:
import getpass
if "GOOGLE_API_KEY" not in os.environ:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Provide your Google API Key: ")

print("API Key is configured.")

## Why RAG became the norm
- You cannot keep fine tuning every time knowledge changes
- Easier updates: change documents, not model weights
- RAG keeps answers grounded in your sources
- Better for enterprise use cases with private data and citations

## RAG, explained as a simple loop
- User asks a question
- System retrieves relevant chunks from your data
- LLM answers using retrieved context
- Goal: accurate answers that cite and stick to the source


![RAG loop visual](images/ragw.drawio.png)

## Baseline RAG demo plan (what we will build now)
- Load a small document set
- Chunk documents
- Create embeddings
- Store in vector DB
- Retrieve top k for a query
- Generate answer using retrieved context

## Key RAG terms you must know
- Chunking: split documents into searchable pieces
- Embeddings: convert text to vectors for similarity search
- Vector store: database optimized for nearest neighbor search
- Retriever: gets top k relevant chunks
- Context assembly: builds the final prompt for the LLM

## Part A: Build the knowledge base (load, chunk, embed, store)

## Optional: skip indexing if DB already exists

In [ ]:
if os.path.exists(CHROMA_DIR) and len(os.listdir(CHROMA_DIR)) > 0:
    print(f"Vector DB already exists at {CHROMA_DIR}. You can safely skip Part A and go right to Part B if you want!")
else:
    print("No existing Vector DB found or it is empty. Please proceed with Part A to create it.")

## Load PDFs from documents folder

In [ ]:
import glob

pdf_files = glob.glob(os.path.join(DOCS_DIR, "*.pdf"))
print(f"Found {len(pdf_files)} PDF files in {DOCS_DIR}\n")

all_docs = []
for file_path in pdf_files:
    print(f"Loading {os.path.basename(file_path)}...")
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    all_docs.extend(docs)
    
print(f"\nTotal pages loaded across all files: {len(all_docs)}")

## Add metadata for traceability

In [ ]:
import pathlib

for doc in all_docs:
    source = doc.metadata.get("source", "Unknown")
    filename = pathlib.Path(source).stem
    doc.metadata["paper_title"] = filename
    
    page_num = doc.metadata.get("page", 0) + 1
    doc.metadata["page_number"] = page_num

if all_docs:
    sample_meta = all_docs[0].metadata
    print("Sample Document Metadata:")
    for k, v in sample_meta.items():
        print(f"  {k}: {v}")

## Chunking decisions that change everything
- Chunk size controls recall vs precision
- Bad chunking is a common reason RAG fails
- Add metadata like title, section, page number
- Overlap helps preserve context across boundaries

## Chunk the documents

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(all_docs)
print(f"Created {len(chunks)} chunks from the documents.")

if chunks:
    sample_chunk = chunks[0]
    print("\n--- Sample Chunk Preview ---")
    print(f"{sample_chunk.page_content[:300]}...")
    print("\n--- Meta Data ---")
    print(sample_chunk.metadata)

## Embeddings and vector storage
- Vector DB stores vectors and metadata and supports top k search
- Embedding model maps text to a vector space
- Retrieval quality depends on embedding choice and indexing settings
- Similar questions and relevant chunks should be close in that space

## Create embeddings and store in Chroma

In [ ]:
import time
import re
from langchain_core.embeddings import Embeddings

def _extract_retry_seconds(err: Exception, default=35):
    s = str(err)
    # matches: retryDelay': '32s'  OR  "Please retry in 32.94s."
    m = re.search(r"retryDelay'\s*:\s*'(\d+)s'", s) or re.search(r"retry in\s+([\d.]+)s", s)
    if not m:
        return default
    return int(float(m.group(1)))

class RateLimitedEmbeddings(Embeddings):
    def __init__(self, base: Embeddings, delay_s: float = 1.5):
        self.base = base
        self.delay_s = delay_s

    def embed_documents(self, texts):
        vectors = []
        for t in texts:
            while True:
                try:
                    vec = self.base.embed_documents([t])[0]  # one request per chunk
                    vectors.append(vec)
                    break
                except Exception as e:
                    wait_s = _extract_retry_seconds(e, default=35)
                    print(f"Rate limit hit. Sleeping {wait_s}s then retrying...")
                    time.sleep(wait_s)

            # wait after EACH embedding
            time.sleep(self.delay_s)

        return vectors

    def embed_query(self, text):
        # keep query fast (no 1.5s sleep here)
        return self.base.embed_query(text)

In [ ]:
from chromadb.config import Settings

base_embeddings = GoogleGenerativeAIEmbeddings(model=GEMINI_EMBED_MODEL)
embeddings = RateLimitedEmbeddings(base_embeddings, delay_s=1.5)

# Try to load existing DB first
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_DIR,
)

existing_count = 0
try:
    existing_count = vectorstore._collection.count()
except Exception:
    existing_count = 0

if existing_count > 0:
    print(f"Chroma DB already has {existing_count} chunks at {CHROMA_DIR}. Skipping re-indexing.")
else:
    if chunks:
        print("No existing chunks found. Indexing chunks into Chroma...")
        vectorstore.add_documents(chunks)
        vectorstore.persist()
        print(f"Success! Stored {vectorstore._collection.count()} chunks in Chroma DB at {CHROMA_DIR}.")
    else:
        print("No chunks found. Vector DB was not created.")

## Part B: Retrieval Augmented Generation (retrieve, prompt, generate)

## Load the existing Chroma DB

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model=GEMINI_EMBED_MODEL)

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings
)
retriever = vectorstore.as_retriever(search_kwargs={"k": TOP_K})
print(f"Loaded Chroma DB with {vectorstore._collection.count()} chunks.")

## Define demo queries

In [ ]:
# Demo queries
#1 active query + 1 follow-up query (commented out).

# Paper 1: Attention Is All You Need (Transformer)
Q1 = "What is the role of positional encoding in the transformer architecture?"

# Paper 2: LoRA (Efficient Fine Tuning)
# Q2 = "How does LoRA reduce the number of trainable parameters compared to full fine tuning?"


demo_queries = [Q1]

In [ ]:
# # Demo queries (2 per paper)
# #1 active query + 1 follow-up query (commented out).

# # Paper 1: Attention Is All You Need (Transformer)
# Q1_FOLLOWUP = "Can you explain it in simpler terms with an example?"

# # Paper 2: LoRA (Efficient Fine Tuning)
# # Q2_FOLLOWUP = "What happens to the frozen pretrained weights during training?"


# demo_queries = [Q1_FOLLOWUP]

## Retrieval: what “top k” really means
- Retriever returns top k chunks by similarity score
- Filters help: metadata filters, document scope, date range
- Garbage in retrieval means garbage out generation
- Reranking can improve ordering using a stronger scoring model

## Retrieve top k chunks and show sources

In [ ]:
import time

retrieval_cache = {}  # maps original query -> list of (Document, score)
timing = {}           # maps original query -> {"retrieval_s": float}

print(f"Using TOP_K = {TOP_K}")

for q in demo_queries:
    print(f"\n{'='*60}\nUSER QUERY: {q}\n{'='*60}")

    
    max_retries_rew = 3
    rewritten_q = q
    for attempt_rew in range(max_retries_rew):
        try:
            rewritten_q = rewrite_query(q)
            break
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                print(f"Rate limited in rewrite, waiting 65s... (attempt {attempt_rew+1})")
                time.sleep(65)
            else:
                raise e

    if rewritten_q != q:
        print(f"Rewritten query: {rewritten_q}")
    else:
        print("Rewritten query: (same as input)")

    t0 = time.time()
    results = vectorstore.similarity_search_with_score(rewritten_q, k=TOP_K)
    t_retrieval = time.time() - t0

    retrieval_cache[q] = results
    timing[q] = {"retrieval_s": t_retrieval}

    print(f"\nTop {TOP_K} Retrieval Results (showing source + page + score):\n")
    for i, (doc, score) in enumerate(results, start=1):
        title = doc.metadata.get("paper_title", "Unknown")
        page = doc.metadata.get("page_number", "?")
        preview = doc.page_content.replace("\n", " ")[:170]
        print(f"[{i}] score={score:.4f} | {title} | page {page}")
        print(f"    {preview}...\n")

print("\nDone. Retrieval results are stored in `retrieval_cache`.")


## Build prompt with memory + retrieved context

In [ ]:
# contextual assembly
rag_prompt_template = PromptTemplate(
    input_variables=["context", "memory", "query"],
    template="""You are a research assistant helping students understand concepts from research articles.
Explain as simply as possible, but include formulas when needed.

Rules:
1) Answer the question using ONLY the provided context chunks.
2) If the context does not contain the answer, say you do not have enough information from the provided papers.
3) Add lightweight citations in your answer in this format: [Paper: <paper_title> Page <page_number>]

--- CONTEXT CHUNKS ---
{context}

--- CONVERSATION HISTORY ---
{memory}

--- USER QUESTION ---
{query}

Write a clear, structured answer.
"""
)


## Call Gemini and update memory

In [ ]:
import time

llm = ChatGoogleGenerativeAI(model=GEMINI_LLM_MODEL, temperature=0.1)

memory["turns"] = []

answers = {}  # maps original query -> generated answer text

for q in demo_queries:
    print(f"\n{'='*70}\nQUESTION: {q}\n{'='*70}")
    results = retrieval_cache.get(q)

    # Safety fallback: if you rerun this cell without rerunning retrieval
    if results is None:
        t0 = time.time()
        results = vectorstore.similarity_search_with_score(rewritten_q, k=TOP_K)
        t_retrieval = time.time() - t0
        retrieval_cache[q] = results
        timing[q] = {"retrieval_s": t_retrieval}

    # Build context (use the retrieved docs, ignore the score for prompt)
    ctx_lines = []
    for doc, score in results:
        title = doc.metadata.get("paper_title", "Unknown")
        page = doc.metadata.get("page_number", "?")
        ctx_lines.append(f"[Paper: {title} Page {page}]\n{doc.page_content}")
    context_text = "\n\n".join(ctx_lines)

    # Build memory text
    if memory["turns"]:
        memory_text = ""
        for turn in memory["turns"]:
            memory_text += f"User: {turn['user']}\nAssistant: {turn['assistant']}\n\n"
    else:
        memory_text = "(No previous history)"

    final_prompt = rag_prompt_template.format(
        context=context_text,
        memory=memory_text,
        query=q
    )

    t0 = time.time()
    
    class MockResponse:
        content = "This is a fallback response due to API rate limits."
    response = None
    max_retries_gen = 3
    for attempt_gen in range(max_retries_gen):
        try:
            response = llm.invoke(final_prompt)
            break
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                print(f"Rate limited in generation, waiting 65s... (attempt {attempt_gen+1})")
                time.sleep(65)
            else:
                raise e
    if response is None:
        response = MockResponse()

    t_gen = time.time() - t0

    answer = response.content.strip()
    answers[q] = answer
    timing[q]["generation_s"] = t_gen
    timing[q]["total_s"] = timing[q]["retrieval_s"] + t_gen

    print(f"\nANSWER:\n{answer}\n")
    print(f"Latency: retrieval={timing[q]['retrieval_s']:.2f}s | generation={t_gen:.2f}s | total={timing[q]['total_s']:.2f}s")

    memory["turns"].append({"user": q, "assistant": answer})
    time.sleep(1)

print("\nDone. Answers are stored in `answers` and memory is updated in `memory`.")


## Metrics we will compare (simple and practical)
- Retrieval hit rate: did we fetch the right chunk
- Faithfulness check: is the answer supported by retrieved text
- Latency: baseline vs rewritten pipeline time
- Recall at k: does the correct source appear in top k

## Quick evaluation (simple workshop metrics)
We keep evaluation lightweight here. The goal is to quickly sanity check retrieval + grounding.


In [ ]:
def _expected_keywords(query: str):
    q = query.lower()
    if "positional" in q or "transformer" in q or "attention" in q:
        return ["attention", "transformer"]
    if "lora" in q:
        return ["lora"]
    if "rag" in q or "retrieval" in q:
        return ["rag", "retrieval"]
    return []

def _hit_and_rank(results, keywords):
    if not keywords:
        return False, None
    for idx, (doc, score) in enumerate(results, start=1):
        title = str(doc.metadata.get("paper_title", "")).lower()
        if any(k in title for k in keywords):
            return True, idx
    return False, None

print("Eval examples:", len(demo_queries))

hits = 0
mrr_sum = 0.0
faithful = 0

for q in demo_queries:
    kws = _expected_keywords(rewritten_q)
    results = retrieval_cache[q]
    hit, rank = _hit_and_rank(results, kws)

    if hit:
        hits += 1
        mrr_sum += 1.0 / rank

    ans = answers.get(q, "")
    has_citation = "[Paper:" in ans
    if has_citation:
        faithful += 1

    top1_title = results[0][0].metadata.get("paper_title", "Unknown")
    print(f"\nQuery: {q}")
    print(f"Expected keywords: {kws if kws else '(none)'}")
    print(f"Top1 source: {top1_title}")
    print(f"Retrieval Hit@{TOP_K}: {hit} | First-hit rank: {rank}")
    print(f"Citation present in answer: {has_citation}")
    print(f"Latency total: {timing[q]['total_s']:.2f}s")

hit_rate = hits / max(len(demo_queries), 1)
mrr = mrr_sum / max(len(demo_queries), 1)
faithfulness_rate = faithful / max(len(demo_queries), 1)

print("\n" + "="*60)
print(f"Retrieval Hit@{TOP_K}: {hit_rate*100:.1f}%")
print(f"MRR@{TOP_K}: {mrr:.3f}")
print(f"Citation presence rate: {faithfulness_rate*100:.1f}%")
print("Note: This is a lightweight sanity check, not a full benchmark.")


## Show memory structure

In [ ]:
import json
print(json.dumps(memory, indent=2))

## Common failure modes of baseline RAG
- Missing the right chunk due to wording mismatch
- Chunks too big or too small
- Embeddings not ideal for your domain
- LLM answers confidently even with weak context
- High latency from too many retrieved chunks

## One simple upgrade: Query rewriting
- Before retrieving, use a small LLM to rewrite the user query
- Goal: make the query more specific and retrieval friendly
- Then retrieve using rewritten query and answer with better context
- Examples: add synonyms, expand acronyms, include key entities

![Query Rewriting](images/query_rewritew.drawio.png)

## TODO: Query rewriting hook in this notebook
For now, `rewrite_query(query)` returns the same query.
Later, you can plug in a small LLM call inside `rewrite_query` to expand acronyms, add synonyms, and make the query more retrieval friendly.


## When to prompt vs fine tune vs RAG
- Prompting: fastest to start, good for general tasks
- Fine tuning: best when you want consistent behavior and format
- RAG: best when you need factual grounding, private docs, or freshness
- In real products, you often combine RAG + light tuning

![When to prompt vs fine tune vs RAG](images/slide_11_img_0.png)

## What is next in RAG (quick tour)
- Hybrid retrieval: BM25 + embeddings together
- Reranking becomes standard for quality
- Better chunking: structure aware splitting, semantic chunking
- Context compression and citations for long documents
- Evaluation driven iteration with test sets and monitoring


## Resources
- Github Repo Link for Part-2 of workshop

## References
- https://jillanisofttech.medium.com/rag-vs-fine-tuning-vs-prompt-engineering-optimizing-ai-llm-models-9073013e0662